In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


# 1. HER DEFINERER VI OPSKRIFTEN PÅ DEN KVADRATISKE NEURON
class QuadraticLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(QuadraticLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Det primære (lineære) led: W1
        self.weight1 = nn.Parameter(torch.Tensor(out_features, in_features))

        # Det kvadratiske led: W2 (Adapteren)
        self.weight2 = nn.Parameter(torch.Tensor(out_features, in_features))

        self.bias = nn.Parameter(torch.Tensor(out_features))

        # Initialisering
        nn.init.kaiming_uniform_(self.weight1)
        nn.init.zeros_(self.weight2)  # Starter som 0, så den lærer gradvist
        nn.init.zeros_(self.bias)

    def forward(self, x):
        # Det normale lineære pas
        linear_term = torch.matmul(x, self.weight1.t())

        # Det kvadratiske pas (x i anden)
        quadratic_term = torch.matmul(x**2, self.weight2.t())

        return linear_term + quadratic_term + self.bias


# 2. HER HENTER VI RESNET-18
# Bemærk: Vi bruger weights=ResNet18_Weights.DEFAULT for at få den præ-trænede version
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


# 3. HER FRYSER VI BACKBONEN
for param in backbone.parameters():
    param.requires_grad = False


# 4. HER BRUGER VI VORES NYE 'QuadraticLinear' TIL AT UDSKIFTE DET SIDSTE LAG
num_ftrs = (
    backbone.fc.in_features
)  # Finder ud af, hvor mange inputs det sidste lag kræver (512)
num_classes = 10  # 10 klasser

# Nu kender Python 'QuadraticLinear', fordi vi definerede den øverst i koden!
backbone.fc = QuadraticLinear(num_ftrs, num_classes)


# 5. TEST AT DET VIRKER
print("Succes! Det sidste lag i ResNet er nu skiftet ud med:")
print(backbone.fc)